# The Text Editor Tool (Lesson 30)

Follow-along notebook for *The text edit tool* in Building with the Claude API.

**The key idea:** the text editor tool's *schema* is built into Claude, but *you* implement what each command does. Claude asks to `view`, `str_replace`, `create`, or `insert`; your code performs the file operation and returns the result.

**Safety:** everything happens inside a `workspace/` folder next to this notebook. The editor refuses any path that would escape it, so Claude can't touch the rest of your machine.

**Run order:** Run All after a kernel restart.

## 1. Setup and chat helpers

In [ ]:
import json
import shutil
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()
MODEL = "claude-haiku-4-5"


def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})


def add_assistant_message(messages, message):
    content = message.content if hasattr(message, "content") else message
    messages.append({"role": "assistant", "content": content})


def chat(messages, system=None, tools=None, max_tokens=4000):
    params = {"model": MODEL, "max_tokens": max_tokens, "messages": messages}
    if system:
        params["system"] = system
    if tools:
        params["tools"] = tools
    return client.messages.create(**params)


def text_from_message(message):
    return "\n".join(b.text for b in message.content if b.type == "text")

## 2. The schema stub

You don't write a JSON schema for this tool, just a small stub with a version `type` and a `name`. Claude expands it into the full tool spec.

The lesson's snippet only covers Claude 3.x models. Current Claude 4+ models (including Haiku 4.5) use `text_editor_20250728` with the name `str_replace_based_edit_tool`. Check the [text editor tool docs](https://platform.claude.com/docs/en/agents-and-tools/tool-use/text-editor-tool) if a newer version appears.

In [ ]:
def get_text_edit_schema(model):
    if model.startswith("claude-3-7-sonnet"):
        return {"type": "text_editor_20250124", "name": "str_replace_editor"}
    if model.startswith("claude-3-5-sonnet"):
        return {"type": "text_editor_20241022", "name": "str_replace_editor"}
    # Claude 4 and later
    return {"type": "text_editor_20250728", "name": "str_replace_based_edit_tool"}


text_edit_schema = get_text_edit_schema(MODEL)
text_edit_schema

## 3. Implementing the commands

Commands Claude can send:

| Command | Inputs | What it does |
|---|---|---|
| `view` | `path`, optional `view_range` | Show a file with line numbers, or list a directory |
| `str_replace` | `path`, `old_str`, `new_str` | Replace text that appears **exactly once** |
| `create` | `path`, `file_text` | Create (or overwrite) a file |
| `insert` | `path`, `insert_line`, `insert_text` | Insert text after a line (0 = top of file) |
| `undo_edit` | `path` | Revert the last edit (older tool versions only) |

Errors are raised as exceptions; the router turns them into `is_error` tool results so Claude can correct itself.

In [ ]:
class TextEditorTool:
    def __init__(self, base_dir="workspace"):
        self.base_dir = Path(base_dir).resolve()
        self.base_dir.mkdir(exist_ok=True)
        self.backups = {}   # path -> list of previous contents, for undo_edit

    # -- path safety: every path must stay inside base_dir ------------------
    def _resolve(self, path):
        cleaned = str(path).strip().lstrip("/\\")
        full = (self.base_dir / cleaned).resolve()
        if full != self.base_dir and self.base_dir not in full.parents:
            raise PermissionError(f"Access denied: {path} is outside the workspace")
        return full

    def _backup(self, full):
        if full.exists():
            self.backups.setdefault(str(full), []).append(full.read_text(encoding="utf-8"))

    # -- commands ------------------------------------------------------------
    def view(self, path, view_range=None):
        full = self._resolve(path)
        if full.is_dir():
            entries = sorted(full.rglob("*"))
            lines = [str(p.relative_to(self.base_dir)) + ("/" if p.is_dir() else "") for p in entries]
            return "\n".join(lines) or "(empty directory)"
        if not full.exists():
            raise FileNotFoundError(f"File not found: {path}")

        lines = full.read_text(encoding="utf-8").splitlines()
        start, end = 1, len(lines)
        if view_range:
            start = max(1, view_range[0])
            end = len(lines) if view_range[1] == -1 else min(len(lines), view_range[1])
        return "\n".join(f"{i}: {lines[i - 1]}" for i in range(start, end + 1))

    def str_replace(self, path, old_str, new_str=""):
        full = self._resolve(path)
        if not full.exists():
            raise FileNotFoundError(f"File not found: {path}")
        content = full.read_text(encoding="utf-8")
        count = content.count(old_str)
        if count == 0:
            raise ValueError("No match found for old_str. Check whitespace and indentation.")
        if count > 1:
            raise ValueError(f"Found {count} matches for old_str. Include more context to make it unique.")
        self._backup(full)
        full.write_text(content.replace(old_str, new_str, 1), encoding="utf-8")
        return "Successfully replaced text at exactly one location."

    def create(self, path, file_text):
        full = self._resolve(path)
        full.parent.mkdir(parents=True, exist_ok=True)
        self._backup(full)
        full.write_text(file_text, encoding="utf-8")
        return f"Successfully created {path}"

    def insert(self, path, insert_line, insert_text):
        full = self._resolve(path)
        if not full.exists():
            raise FileNotFoundError(f"File not found: {path}")
        lines = full.read_text(encoding="utf-8").splitlines(keepends=True)
        if not 0 <= insert_line <= len(lines):
            raise ValueError(f"insert_line {insert_line} is out of range (0-{len(lines)})")
        if lines and not lines[-1].endswith("\n"):
            lines[-1] += "\n"
        new_text = insert_text if insert_text.endswith("\n") else insert_text + "\n"
        self._backup(full)
        lines.insert(insert_line, new_text)
        full.write_text("".join(lines), encoding="utf-8")
        return f"Successfully inserted text after line {insert_line}"

    def undo_edit(self, path):
        full = self._resolve(path)
        history = self.backups.get(str(full))
        if not history:
            raise ValueError(f"No edits to undo for {path}")
        full.write_text(history.pop(), encoding="utf-8")
        return f"Successfully reverted last edit to {path}"


editor = TextEditorTool("workspace")
print("Workspace:", editor.base_dir)

## 4. Router and conversation loop

In [ ]:
def run_text_editor(tool_input):
    command = tool_input["command"]
    path = tool_input["path"]

    if command == "view":
        return editor.view(path, tool_input.get("view_range"))
    if command == "str_replace":
        return editor.str_replace(path, tool_input["old_str"], tool_input.get("new_str", ""))
    if command == "create":
        return editor.create(path, tool_input["file_text"])
    if command == "insert":
        # newer tool versions send insert_text; older ones sent new_str
        text = tool_input.get("insert_text", tool_input.get("new_str", ""))
        return editor.insert(path, tool_input["insert_line"], text)
    if command == "undo_edit":
        return editor.undo_edit(path)
    raise ValueError(f"Unknown command: {command}")


def run_tool(tool_name, tool_input):
    if tool_name in ("str_replace_based_edit_tool", "str_replace_editor"):
        return run_text_editor(tool_input)
    raise ValueError(f"Unknown tool: {tool_name}")


def run_tools(message):
    results = []
    for block in message.content:
        if block.type != "tool_use":
            continue
        print(f"  -> {block.input.get('command')} {block.input.get('path')}")
        try:
            output = run_tool(block.name, block.input)
            results.append({"type": "tool_result", "tool_use_id": block.id,
                            "content": output, "is_error": False})
        except Exception as e:
            print(f"     error: {e}")
            results.append({"type": "tool_result", "tool_use_id": block.id,
                            "content": f"Error: {e}", "is_error": True})
    return results


def run_conversation(messages, tools, max_turns=15):
    for _ in range(max_turns):
        response = chat(messages, tools=tools)
        add_assistant_message(messages, response)

        text = text_from_message(response)
        if text:
            print(f"Claude: {text}\n")

        if response.stop_reason != "tool_use":
            break
        add_user_message(messages, run_tools(response))
    return messages

## 5. Test the editor directly (no API calls)
Worth doing before handing it to Claude.

In [ ]:
editor.create("scratch.txt", "line one\nline two\n")
editor.insert("scratch.txt", 1, "inserted after line one")
editor.str_replace("scratch.txt", "line two", "line TWO")
print(editor.view("scratch.txt"))

try:
    editor.view("../.env")          # should be blocked
except PermissionError as e:
    print("Blocked:", e)

(editor.base_dir / "scratch.txt").unlink()

## 6. Seed a starter `main.py`
The lesson's examples assume a `main.py` exists.

In [ ]:
editor.create("main.py", """def greet(name):
    return f"Hello, {name}!"


def add(a, b):
    return a + b


if __name__ == "__main__":
    print(greet("world"))
    print(add(2, 3))
""")
print(editor.view("."))

## 7. Example 1: read and summarize

In [ ]:
messages = []
add_user_message(messages, "Open the ./main.py file and summarize its contents.")
run_conversation(messages, tools=[text_edit_schema]);

## 8. Example 2: modify a file and create a test file

In [ ]:
messages = []
add_user_message(
    messages,
    "Open the ./main.py file and write out a function to calculate pi to the 5th digit. "
    "Then create a ./test.py file to test your implementation.",
)
run_conversation(messages, tools=[text_edit_schema]);

## 9. Review what Claude wrote

In [ ]:
for name in ("main.py", "test.py"):
    print(f"===== {name} =====")
    try:
        print(editor.view(name))
    except FileNotFoundError as e:
        print(e)
    print()

Once you've read the code and are happy with it, you can run the tests yourself. Running code a model wrote is your call, which is why this cell is commented out:

In [ ]:
# !cd workspace && python test.py

## Reset the workspace
Run this to clear `workspace/` and start over.

In [ ]:
# shutil.rmtree(editor.base_dir); editor = TextEditorTool("workspace")

## Takeaways
- Claude knows the tool's schema; you only send a versioned stub, but you **implement every command**.
- Your implementation is the security boundary: sandbox paths, validate inputs, and return clear errors.
- `str_replace` requires a unique match, which is why Claude usually `view`s a file before editing it.
- Add `workspace/` to `.gitignore` if you don't want Claude's scratch files in your repo.